# Collecting Data from APIs

Inspect JSON responses, send reliable HTTP requests, and flatten nested data into tables.

> 🧭 This notebook focuses on practical data work. Examples are intentionally compact and do not cover machine learning.

## Learning goals
- Understand the API request/response cycle and JSON payloads.
- Use `requests` with timeouts and status-code checks.
- Convert nested JSON to a DataFrame with pandas.
- Keep credentials out of notebooks and handle pagination deliberately.

The examples run offline using a representative response. The live-request example is intentionally optional so the notebook remains reproducible without internet access.

In [1]:
import pandas as pd
import requests

sample_payload = {
    "results": [
        {"id": 1, "name": "Library A", "location": {"city": "Baku", "district": "Central"}, "visits": 120},
        {"id": 2, "name": "Library B", "location": {"city": "Ganja", "district": "West"}, "visits": 85},
    ],
    "page": 1,
    "total_pages": 1,
}
records = pd.json_normalize(sample_payload["results"])
display(records)

,id,name,visits,location.city,location.district
0,1,Library A,120,Baku,Central
1,2,Library B,85,Ganja,West


## A reliable request pattern
Always set a timeout, check the HTTP status, and validate the response shape before trusting the data. APIs can rate-limit requests or change their schemas.

In [2]:
def fetch_json(url, params=None, timeout=10):
    """Fetch JSON from a trusted endpoint; raises clear errors on failure."""
    response = requests.get(url, params=params, timeout=timeout)
    response.raise_for_status()
    payload = response.json()
    if not isinstance(payload, (dict, list)):
        raise ValueError("Expected a JSON object or array")
    return payload

# Optional live request. Uncomment only when you have internet access and
# have reviewed the endpoint's documentation and usage policy.
# payload = fetch_json("https://api.example.com/records", params={"limit": 100})
# display(pd.json_normalize(payload["results"]))
print("Function defined; no network request was made.")

Function defined; no network request was made.


## Flatten nested JSON and preserve provenance
Nested keys become columns such as `location.city`. Keep source URL, access date, query parameters, and pagination metadata in your analysis notes so the collection can be reproduced.

In [3]:
flat = pd.json_normalize(sample_payload["results"], sep="_")
flat["source"] = "offline example payload"
flat["retrieved_at_utc"] = pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%d %H:%M:%S%z")
display(flat)

# Check for an expected schema before downstream work.
required = {"id", "name", "location_city", "visits"}
missing_columns = required.difference(flat.columns)
if missing_columns:
    raise ValueError(f"Unexpected API schema; missing: {sorted(missing_columns)}")
print("Schema check passed")

Schema check passed


,id,name,visits,location_city,location_district,source,retrieved_at_utc
0,1,Library A,120,Baku,Central,offline example payload,2026-10-19 10:49:41+0000
1,2,Library B,85,Ganja,West,offline example payload,2026-10-19 10:49:41+0000


## Pagination, secrets, and failures
- Read the API documentation to understand page tokens, rate limits, and retry guidance.
- Keep API keys in environment variables (for example `os.environ["SERVICE_API_KEY"]`), not in source code.
- Treat status `401/403` as authentication/permission problems, `404` as a missing resource, and `429` as rate limiting.
- Do not retry every error blindly; use bounded retries and respect `Retry-After` when provided.

## Practice
Given a response dictionary with a `results` list, convert the nested `owner.name` field to a column and keep only rows with an active status.

In [4]:
payload = {"results": [
    {"id": 1, "owner": {"name": "Ari"}, "active": True},
    {"id": 2, "owner": {"name": "Bea"}, "active": False},
]}
result = pd.json_normalize(payload["results"], sep="_")
display(result.loc[result["active"], ["id", "owner_name"]])

,id,owner_name
0,1,Ari
